To accomplish this in Python, you'll want to use the Box SDK to connect to your Box folder, walk the entire folder tree, and extract metadata for each file. Here's a complete outline and sample code to generate a table (Pandas DataFrame) with the following columns:

file_path

extension

size_bytes

created_at

modified_at

file_id (optional, useful for traceability or future downloads)

✅ Step-by-step Plan
Authenticate with Box API using JWT or OAuth2.

Recursively walk through all folders starting from your root folder ID.

For every file:

Capture full path

Extract extension

Get file size

Get metadata (created, modified)

Store in a list and convert to a Pandas DataFrame.

In [ ]:
# !pip install boxsdk pandas


1. Create a Box Developer Account
If you don’t already have one:

Go to: https://app.box.com/developers/console

Log in with your Box account

2. Create a New App
Click "Create New App"

Choose "Custom App"

Choose "OAuth 2.0 with JWT (Server Authentication)"

Give your app a name, e.g., box-directory-scanner

Click "Create App"

3. Generate and Download box_config.json
Inside your new app, go to the Configuration tab

Scroll to App Settings

Click "Generate a Developer Token" (for temporary testing) — optional

Under "App Credentials", click "Download Config" — this is your box_config.json

This file contains:

Client ID

Client Secret

Public/Private Key Pair

Enterprise ID

JWT settings

4. Enable App Authorization
Scroll down to "App Access Level", make sure it's set to App + Enterprise Access

Under "Application Scopes", check:

Read and write all files and folders stored in Box

Manage users (optional for some actions)

Click "Save Changes"

5. Submit App for Authorization
Scroll to "App Authorization"

Click "Submit" for your Box Admin to approve your app

If you're the Admin, you can approve it immediately via the Admin Console > Apps > Custom Apps



In [ ]:
# Step 1: Authenticate using Developer Token

from boxsdk import OAuth2, Client

#Regenerate Developer Token here: https://uofi.app.box.com/developers/console/app/2375273/configuration
developer_token = 'AQWytDkbUOSJO2V4DGyQ31RLxKDu20eH'

# Authenticate with Developer Token
auth = OAuth2(client_id=None, client_secret=None, access_token=developer_token)
client = Client(auth)

# Test connection
user = client.user().get()
print(f'Successfully connected as: {user.name}')


In [ ]:
me = client.user().get()
print(f"Developer token belongs to: {me.name} <{me.login}>")


In [ ]:
def list_files_recursive(folder, parent_path=''):
    file_info = []
    has_content = False  # Flag to detect if the folder has any items

    for item in folder.get_items():
        has_content = True
        full_path = os.path.join(parent_path, item.name)

        if item.type == 'folder':
            file_info.extend(list_files_recursive(client.folder(item.id), full_path))
        elif item.type == 'file':
            file_metadata = client.file(item.id).get()
            file_info.append({
                'file_path': full_path,
                'extension': os.path.splitext(item.name)[1].lower(),
                'size_bytes': file_metadata.size,
                'created_at': file_metadata.created_at,
                'modified_at': file_metadata.modified_at,
                'file_id': file_metadata.id,
                'is_empty': False
            })

    # If folder has no files or subfolders, record it as empty
    if not has_content:
        file_info.append({
            'file_path': parent_path,
            'extension': None,
            'size_bytes': None,
            'created_at': None,
            'modified_at': None,
            'file_id': None,
            'is_empty': True
        })

    return file_info

import os
import pandas as pd

target_folder_id = '318345592147'
root_folder = client.folder(folder_id=target_folder_id)
all_files = list_files_recursive(root_folder)
df = pd.DataFrame(all_files)


In [ ]:
import os
import pandas as pd

target_folder_id = '318345592147'
root_folder = client.folder(folder_id=target_folder_id)
all_files = list_files_recursive(root_folder)
df = pd.DataFrame(all_files)



In [ ]:
df

In [ ]:
df_empty = df[df["is_empty"] == True]
df_empty

In [ ]:
df_nonempty = df[df["is_empty"] == False]
df_nonempty

In [ ]:
df_jpg = df[df["extension"] == ".jpg"]
df_jpg

In [ ]:
# from boxsdk import OAuth2, Client

# auth = OAuth2(client_id=None, client_secret=None, access_token=developer_token)
# client = Client(auth)

# # 2. Define file ID and local download path
# file_id = '1844528593445'
# output_path = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads\image_file_1844528593445.jpg'  # You can customize the file name

# # 3. Download the file
# with open(output_path, 'wb') as f:
#     client.file(file_id).download_to(f)

# print(f"File downloaded to: {output_path}")


In [ ]:
df["extension"].value_counts()

## 📂 File Conversion to Text — Overview

This section outlines how to convert various file types into extractable text using Python. Different methods are recommended depending on file type.

---

### ✅ File Types and Their Conversion Methods

| Extension(s)                             | Description                          | Text Extraction Method                                                                 |
|------------------------------------------|--------------------------------------|----------------------------------------------------------------------------------------|
| `.pdf`, `pdf1`, `pdf3`, `pd`             | PDF documents                        | `PyMuPDF`, `pdfplumber`, or `PyPDF2` (text-based) <br> `pytesseract` + `pdf2image` (image-based) |
| `.jpg`, `.png`, `.gif`, `.tif`, `.eps`   | Images                               | OCR using `pytesseract`                                                               |
| `.doc`, `.docx`, `doc1`, `doc2`, `doc5`, `do` | Word documents                  | `python-docx` for `.docx`, `textract` or `antiword` for `.doc`                        |
| `.xls`, `.xlsx`, `xls2`                  | Excel spreadsheets                   | `pandas.read_excel()`, `openpyxl`, or `xlrd`                                          |
| `.ppt`, `.ppt1`                          | PowerPoint presentations             | `python-pptx`                                                                          |
| `.eml`, `mbox`                           | Email files and archives             | `email` module, `extract_msg`, or `mailbox`                                           |
| `.mdb`                                   | Microsoft Access DB                  | `pyodbc` or `mdbtools`                                                                |
| `.db`                                    | SQLite databases                     | `sqlite3`, SQL queries                                                                |
| `.mhtml`, `.mht`, `.htm`                 | Archived or plain HTML               | `BeautifulSoup` or `html2text`                                                        |
| `.mov`                                   | Video files                          | Use `ffmpeg` to extract audio/subtitles or take screenshots + OCR                     |
| `.rtf`                                   | Rich Text Format                     | `pypandoc` or convert to `.txt` using `unrtf`                                          |
| `.zip`                                   | Compressed archives                  | Use `zipfile` to extract contents                                                     |
| `.txt`                                   | Plaintext                            | Direct read via `open()`                                                              |
| `.gz`                                    | Compressed files                     | Use `gzip` module to decompress, then read                                            |
| `.toc`, `d1`, `d`, `career2`, `hanrttydoc`, `italics`, `edu'sconflictedcopy2011-12-09)`, `lagrangian`, `particleturbulence`, `` | Unknown/custom labels | Try file content inspection or ignore if unreadable                                    |
| `00361-00850`, `00136-01590`, `00106-00790`, `00111-00480`, `00701-01750`, `00701-01000` | Likely custom batch labels          | May refer to scanned pages—try `OCR` or skip                                          |
| `net3852448d`, `net0e594322`, `dmdelive13b254ad` | Corrupted or malformed              | Try filename cleaning + guessing format                                               |

---

# Box File Downloader Script

This script connects to Box using a developer token, loops through a DataFrame of files, and downloads each file to a local directory. It assumes that the DataFrame `df` already exists and contains file paths, file IDs, and file extensions.

---

## 1. Setup

- Authenticate with the Box API using a developer token.
- Define the local folder where the files will be downloaded.

---

##  2. Filename Extraction Function

- Cleans the filename from the `file_path` column by removing spaces, slashes, and colons.
- Returns a sanitized base filename without its extension.

---

## 3. File Download Loop

- Iterates over each row in the DataFrame where a `file_id` is present.
- Constructs a clean output filename and determines the appropriate file extension.
- Downloads the file using the Box SDK and saves it to the defined local directory.
- Logs a success message or prints an error if the download fails.

---

## Summary

This script is useful for programmatically downloading Box files based on metadata stored in a DataFrame. It ensures that filenames are safe for the local filesystem and that files are consistently saved with the correct extension.


In [ ]:
# import os
# from boxsdk import OAuth2, Client
# import pandas as pd

# # === 1. Setup ===
# developer_token = 'MYdl6CwtuumYitmYYvePSoUNPWMv10AO'
# auth = OAuth2(client_id=None, client_secret=None, access_token=developer_token)
# client = Client(auth)

# # Your existing dataframe `df` is assumed to exist and includes:
# # - file_path
# # - file_id
# # - extension

# # Folder to download to
# download_dir = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads'
# os.makedirs(download_dir, exist_ok=True)

# # === 2. Function to sanitize and extract filename ===
# def extract_filename(file_path):
#     base_name = os.path.basename(file_path)
#     base_name = base_name.replace(" ", "_").replace(":", "_").replace("/", "_")
#     return os.path.splitext(base_name)[0]  # without extension

# # === 3. Loop and download files ===
# for idx, row in df[df["file_id"].notna()].iterrows():
#     try:
#         file_id = row['file_id']
#         extension = row['extension'] or ''
#         extension = extension.strip().replace(" ", "")  # clean weird ones
#         extension = extension if extension.startswith('.') else f".{extension}"

#         base_filename = extract_filename(row['file_path'])
#         output_filename = f"{base_filename}_{file_id}{extension}"
#         output_path = os.path.join(download_dir, output_filename)

#         # Download from Box
#         with open(output_path, 'wb') as f:
#             client.file(file_id).download_to(f)

#         print(f"✅ Downloaded: {output_filename}")

#     except Exception as e:
#         print(f"❌ Failed to download file_id {file_id}: {e}")


In [ ]:
import os
import pandas as pd

# Set the path to the folder containing the downloaded files
download_dir = r'C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\temp_downloads'

# Initialize list to hold file metadata
file_data = []

# Loop through all files in the directory
for root, _, files in os.walk(download_dir):
    for file in files:
        full_path = os.path.join(root, file)
        file_name = os.path.basename(file)
        file_id = os.path.splitext(file_name)[0].split("_")[-1]  # extract the last underscore-separated part
        extension = os.path.splitext(file_name)[1].replace('.', '')  # get extension without the dot
        file_data.append({
            'file_path': full_path,
            'file_name': file_name,
            'file_id': file_id,
            'extension': extension
        })

# Create a DataFrame
df_files = pd.DataFrame(file_data)

# Display the DataFrame
df_files


In [ ]:
df_pdf = df_files[df_files["extension"]=="pdf"]
df_pdf

In [ ]:
from PyPDF2 import PdfReader

def extract_text_from_pdf(filepath):
    try:
        reader = PdfReader(filepath)
        return "\n".join([page.extract_text() or "" for page in reader.pages])
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return ""

# Apply to your df_pdf DataFrame
df_pdf["extracted_text"] = df_pdf["file_path"].apply(extract_text_from_pdf)
df_pdf
# Optional: preview the result
print(df_pdf[['file_name', 'file_id', 'extracted_text']].head())


In [ ]:
df_pdf.to_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\pdf_df\df_pdf.csv")

import pandas as pd

# --- Read the CSV back into a DataFrame ---
df_pdf = pd.read_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\pdf_df\df_pdf.csv")

# --- Print it to check ---
print(df_pdf.head())


In [ ]:
df_pdf["extracted_text"]

In [ ]:
import re

# 1. Define cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Replace multiple spaces/newlines with a single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# 2. Define token counting function (preserve punctuation, split by spaces)
def count_tokens(text):
    if not isinstance(text, str):
        return 0
    tokens = text.split()  # Split by whitespace, keep punctuation
    return len(tokens)

# 3. Apply to your df_pdf
df_pdf["clean_text"] = df_pdf["extracted_text"].apply(clean_text)
df_pdf["token_count"] = df_pdf["clean_text"].apply(count_tokens)

df_pdf


In [ ]:
from nltk.tokenize import sent_tokenize

# Function to count sentences
def count_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(sent_tokenize(text))

# Apply to DataFrame
df_pdf["sentence_count"] = df_pdf["clean_text"].apply(count_sentences)

# Optional: preview



In [ ]:
df_pdf.head()

In [ ]:
# Get descriptive statistics
stats = df_pdf[["sentence_count", "token_count"]].agg(["min", "max", "mean", "std"])

# Display nicely
print(stats)

In [ ]:
from nltk.tokenize import sent_tokenize
import pandas as pd

from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

# --- summarization logic already provided ---
def summarize_sentence_transformers(text, num_sentences=250):
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        return ""
    text = text.strip()
    sentences = sent_tokenize(text)
    if len(sentences) <= num_sentences:
        return text

    embeddings = model.encode(sentences, convert_to_tensor=True)
    scores = util.pytorch_cos_sim(embeddings, embeddings).mean(dim=1)
    top_indices = scores.argsort(descending=True)[:num_sentences]
    summary = " ".join([sentences[i] for i in sorted(top_indices.tolist())])
    return summary

# --- 1. Summarize if needed ---
def conditional_summarize(row, threshold=250):
    if row['sentence_count'] > threshold:
        return summarize_sentence_transformers(row['clean_text'])
    return row['clean_text']

df_pdf["summarized_text"] = df_pdf.apply(conditional_summarize, axis=1)

# --- 2. Sentence and token counts for summarized_text ---
def count_sentences(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(sent_tokenize(text))

def count_tokens(text):
    if not isinstance(text, str) or not text.strip():
        return 0
    return len(text.split())

df_pdf["summarized_sentence_count"] = df_pdf["summarized_text"].apply(count_sentences)
df_pdf["summarized_token_count"] = df_pdf["summarized_text"].apply(count_tokens)

# Optional: preview final columns
print(df_pdf[[
    "file_name", 
    "sentence_count", "token_count",
    "summarized_sentence_count", "summarized_token_count"
]].head())


In [ ]:
# Optional: preview final columns
df_pdf[[
    "file_name", 
    "sentence_count", "token_count",
    "summarized_sentence_count", "summarized_token_count"
]].head()

In [ ]:
df_pdf["summarized_text"]

In [ ]:
df_pdf = df_pdf.reset_index(drop=True)

for i in range(10):
    print(df_pdf["summarized_text"].iloc[i])


In [ ]:
df_pdf.to_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\pdf_df\df_pdf_sentran_summarized.csv")

import pandas as pd

# --- Read the CSV back into a DataFrame ---
df_pdf = pd.read_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\pdf_df\df_pdf_sentran_summarized.csv")



In [ ]:
df_pdf

In [ ]:
import re
import unicodedata

def clean_summarized_text(text):
    if pd.isna(text) or not isinstance(text, str):
        return ""

    # Normalize Unicode artifacts
    text = unicodedata.normalize("NFKC", text)

    # Remove junk copyright footers
    text = re.sub(r"XML Typescript ©.*?Services\.", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Published by the American Institute of Physics.*?", "", text, flags=re.IGNORECASE)

    # Remove "Page X of Y" markers
    text = re.sub(r"Page\s\d+\s+of\s+\d+", "", text, flags=re.IGNORECASE)

    # Remove junk like "Figure 3", "Chapter 5", "Equation (2.9)", "Equation Chapter"
    text = re.sub(r"(Figure|Chapter|Equation)\s+\d+[^\w]*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"Equation\sChapter.*?(?=\d|\Z)", "", text, flags=re.IGNORECASE)

    # Remove weird artifacts like sequences of symbols
    text = re.sub(r"[\*#\{\}\[\]=@]+", " ", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)  # Remove non-ASCII leftovers carefully

    # Collapse multiple newlines into single paragraph breaks
    text = re.sub(r"\n{2,}", "\n", text)

    # Replace long whitespace stretches with single space
    text = re.sub(r" {2,}", " ", text)

    # Trim final text
    text = text.strip()

    return text

df_pdf["summarized_text_cleaned"] = df_pdf["summarized_text"].apply(clean_summarized_text)
df_pdf.to_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\pdf_df\df_pdf_sentran_summarized_cleaned.csv")

In [ ]:
import pandas as pd

# --- Read the CSV ---
df_pdf = pd.read_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\pdf_df\df_pdf_sentran_summarized_cleaned.csv")

# --- Drop columns that start with "Unnamed:" ---
df_pdf = df_pdf.loc[:, ~df_pdf.columns.str.startswith('Unnamed:')]

df_pdf


In [ ]:
# import subprocess
# import pandas as pd
# import os

# # --- Setup ---
# ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
# model_name = "llama3.2"
# output_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction.parquet"
# batch_size = 3

# # --- Define function to call Llama 3.2 model ---
# def call_llama_summarize(text):
#     prompt = f"Summarize this text:\n\n{text}"

#     result = subprocess.run(
#         [ollama_path, "run", model_name, prompt],
#         stdout=subprocess.PIPE,
#         stderr=subprocess.PIPE,
#         text=True,
#         encoding="utf-8"
#     )

#     if result.returncode != 0:
#         print("Error:", result.stderr)
#         return ""

#     return result.stdout.strip()

# # --- Load or create dataframe ---
# if os.path.exists(output_path):
#     df_pdf = pd.read_parquet(output_path)
#     print("Loaded existing file.")
# else:
#     df_pdf["summary"] = ""
#     print("Starting fresh.")

# # --- Ensure new column exists ---
# if "summary" not in df_pdf.columns:
#     df_pdf["summary"] = ""

# # --- Find where to resume ---
# start_idx = df_pdf[df_pdf["summary"] == ""].index.min()
# if pd.isna(start_idx):
#     print("All rows already processed.")
#     start_idx = len(df_pdf)
# else:
#     print(f"Resuming from index {start_idx}.")

# # --- Processing Loop ---
# batch_counter = 0

# for idx in range(start_idx, len(df_pdf)):
#     row = df_pdf.loc[idx]
#     text_to_summarize = row["summarized_text_cleaned"]

#     if pd.notna(text_to_summarize) and isinstance(text_to_summarize, str) and text_to_summarize.strip():
#         summary = call_llama_summarize(text_to_summarize)
#         df_pdf.at[idx, "summary"] = summary
#         batch_counter += 1

#     # Save after every batch_size rows
#     if batch_counter >= batch_size:
#         df_pdf.to_parquet(output_path, index=False)
#         print(f"Saved after processing up to index {idx}.")
#         batch_counter = 0

# # --- Final save after loop finishes ---
# df_pdf.to_parquet(output_path, index=False)
# print("Final save complete.")


In [ ]:
# df_pdf.to_parquet(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction.parquet")

In [ ]:
# import subprocess
# import pandas as pd
# import os

# import pandas as pd

# # --- Read the CSV ---
# df_pdf = pd.read_csv(r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\pdf_df\df_pdf_sentran_summarized_cleaned.csv")

# # --- Drop columns that start with "Unnamed:" ---
# df_pdf = df_pdf.loc[:, ~df_pdf.columns.str.startswith('Unnamed:')]




# # --- Setup ---
# ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
# model_name = "llama3.2"
# output_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction.parquet"
# batch_size = 3

# # --- Define function to call Llama 3.2 model ---
# def call_llama_summarize(text):
#     prompt = f"Summarize this text:\n\n{text}"

#     result = subprocess.run(
#         [ollama_path, "run", model_name, prompt],
#         stdout=subprocess.PIPE,
#         stderr=subprocess.PIPE,
#         text=True,
#         encoding="utf-8"
#     )

#     if result.returncode != 0:
#         print("Error:", result.stderr)
#         return ""

#     return result.stdout.strip()

# # --- Load or create dataframe ---
# if os.path.exists(output_path):
#     df_pdf = pd.read_parquet(output_path)
#     print("Loaded existing file.")
# else:
#     df_pdf = pd.DataFrame(columns=["summarized_text_cleaned", "summary", "processed"])
#     print("Starting fresh.")

# # --- Ensure new columns exist ---
# for col in ["summary", "processed"]:
#     if col not in df_pdf.columns:
#         if col == "processed":
#             df_pdf[col] = False  # Mark as not processed yet
#         else:
#             df_pdf[col] = ""

# # --- Find where to resume ---
# start_idx = df_pdf[df_pdf["processed"] == False].index.min()
# if pd.isna(start_idx):
#     print("All rows already processed.")
#     start_idx = len(df_pdf)
# else:
#     print(f"Resuming from index {start_idx}.")

# # --- Processing Loop ---
# batch_counter = 0

# for idx in range(start_idx, len(df_pdf)):
#     row = df_pdf.loc[idx]
#     text_to_summarize = row["summarized_text_cleaned"]

#     if pd.notna(text_to_summarize) and isinstance(text_to_summarize, str):
#         summary = call_llama_summarize(text_to_summarize)
#         df_pdf.at[idx, "summary"] = summary
#         df_pdf.at[idx, "processed"] = True  # Mark this row as processed
#         batch_counter += 1

#     # Save after every batch_size rows
#     if batch_counter >= batch_size:
#         df_pdf.to_parquet(output_path, index=False)
#         print(f"Saved after processing up to index {idx}.")
#         batch_counter = 0

# # --- Final save after loop finishes ---
# df_pdf.to_parquet(output_path, index=False)
# print("Final save complete.")


In [ ]:
import subprocess
import pandas as pd
import os

# --- Setup ---
ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
model_name = "llama3.2"
input_csv_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\pdf_df\df_pdf_sentran_summarized_cleaned.csv"
output_parquet_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction.parquet"
batch_size = 3

# --- Define function to call Llama 3.2 model ---
def call_llama_summarize(text):
    prompt = f"Summarize this text:\n\n{text}"

    result = subprocess.run(
        [ollama_path, "run", model_name],
        input=prompt,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8"
    )

    if result.returncode != 0:
        print("Error:", result.stderr)
        return ""

    return result.stdout.strip()


# --- Load or create dataframe ---
if os.path.exists(output_parquet_path):
    df_pdf = pd.read_parquet(output_parquet_path)
    print("Loaded existing processed file.")
else:
    df_pdf = pd.read_csv(input_csv_path)
    df_pdf["summary"] = ""
    df_pdf["processed"] = False
    print("Loaded fresh input CSV.")

# --- Ensure required columns ---
for col in ["summary", "processed"]:
    if col not in df_pdf.columns:
        df_pdf[col] = "" if col == "summary" else False

# --- Fix rows that have no text: mark them processed ---
for idx, row in df_pdf[df_pdf["processed"] == False].iterrows():
    text = row["summarized_text_cleaned"]
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        df_pdf.at[idx, "processed"] = True

# --- Find where to resume ---
start_idx = df_pdf[df_pdf["processed"] == False].index.min()
if pd.isna(start_idx):
    print("All rows already processed.")
    start_idx = len(df_pdf)
else:
    print(f"Resuming from index {start_idx}.")

# --- Processing Loop ---
batch_counter = 0

for idx in range(start_idx, len(df_pdf)):
    row = df_pdf.loc[idx]
    text_to_summarize = row["summarized_text_cleaned"]

    if pd.notna(text_to_summarize) and isinstance(text_to_summarize, str) and text_to_summarize.strip():
        summary = call_llama_summarize(text_to_summarize)
        df_pdf.at[idx, "summary"] = summary
        df_pdf.at[idx, "processed"] = True
        batch_counter += 1

    # Save after every batch_size rows
    if batch_counter >= batch_size:
        df_pdf.to_parquet(output_parquet_path, index=False)
        print(f"Saved after processing up to index {idx}.")
        batch_counter = 0

# --- Final save ---
df_pdf.to_parquet(output_parquet_path, index=False)
print("Final save complete.")


In [ ]:
import subprocess
import pandas as pd
import os

# --- Setup ---
ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
model_name = "llama3.2"
input_parquet_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction.parquet"
output_parquet_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction_1.parquet"
batch_size = 3

# --- Define function to create a title from a summary ---
def call_llama_title(summary_text):
    prompt = (
        f"Generate a one-sentence title that describes the following text. "
        f"ONLY output the title and nothing else. No introductions, no commentary, only the title.\n\n{summary_text}"
    )

    result = subprocess.run(
        [ollama_path, "run", model_name],
        input=prompt,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8"
    )

    if result.returncode != 0:
        print("Error:", result.stderr)
        return ""

    output = result.stdout.strip()

    # Keep only the first non-empty line
    lines = [line.strip() for line in output.splitlines() if line.strip()]
    if lines:
        title = lines[0]
    else:
        title = ""

    return title

# --- Load input dataframe ---
if os.path.exists(input_parquet_path):
    df_pdf = pd.read_parquet(input_parquet_path)
    print("Loaded input file with summaries.")
else:
    raise FileNotFoundError("Input parquet file not found.")

# --- Ensure 'summary_title' and 'title_processed' columns exist ---
if "summary_title" not in df_pdf.columns:
    df_pdf["summary_title"] = ""
if "title_processed" not in df_pdf.columns:
    df_pdf["title_processed"] = False

# --- Skip rows with no summary or already processed ---
for idx, row in df_pdf[df_pdf["title_processed"] == False].iterrows():
    summary = row["summary"]
    if pd.isna(summary) or not isinstance(summary, str) or summary.strip() == "":
        df_pdf.at[idx, "title_processed"] = True

# --- Find where to resume ---
start_idx = df_pdf[df_pdf["title_processed"] == False].index.min()
if pd.isna(start_idx):
    print("All titles already processed.")
    start_idx = len(df_pdf)
else:
    print(f"Resuming title generation from index {start_idx}.")

# --- Processing Loop ---
batch_counter = 0

for idx in range(start_idx, len(df_pdf)):
    row = df_pdf.loc[idx]
    summary = row["summary"]

    if pd.notna(summary) and isinstance(summary, str) and summary.strip():
        title = call_llama_title(summary)
        df_pdf.at[idx, "summary_title"] = title
        df_pdf.at[idx, "title_processed"] = True
        batch_counter += 1

    if batch_counter >= batch_size:
        df_pdf.to_parquet(output_parquet_path, index=False)
        print(f"Saved after title generation up to index {idx}.")
        batch_counter = 0

# --- Final Save ---
df_pdf.to_parquet(output_parquet_path, index=False)
print("Final save with titles complete.")


In [ ]:
import subprocess
import pandas as pd
import os

# --- Setup ---
ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
model_name = "llama3.2"
input_parquet_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction_1.parquet"
output_parquet_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction_2.parquet"
batch_size = 3

# --- Define function to detect document type ---
def call_llama_doc_type(summary_text):
    prompt = (
        f"Analyze the following text and tell me the type of document it is. "
        f"Be specific (e.g., 'email between X and Y about...', 'research paper on...', 'technical formula sheet about...', etc.). "
        f"ONLY output the type of document as a single sentence. Do NOT include any introductions, explanations, or additional text.\n\n"
        f"{summary_text}"
    )

    result = subprocess.run(
        [ollama_path, "run", model_name],
        input=prompt,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8"
    )

    if result.returncode != 0:
        print("Error:", result.stderr)
        return ""

    output = result.stdout.strip()

    # Keep only the first non-empty line
    lines = [line.strip() for line in output.splitlines() if line.strip()]
    if lines:
        doc_type = lines[0]
    else:
        doc_type = ""

    return doc_type

# --- Load input dataframe ---
if os.path.exists(input_parquet_path):
    df_pdf = pd.read_parquet(input_parquet_path)
    print("Loaded input file with summaries.")
else:
    raise FileNotFoundError("Input parquet file not found.")

# --- Ensure new columns exist ---
if "document_type" not in df_pdf.columns:
    df_pdf["document_type"] = ""
if "doc_type_processed" not in df_pdf.columns:
    df_pdf["doc_type_processed"] = False

# --- Skip rows with no summary or already processed ---
for idx, row in df_pdf[df_pdf["doc_type_processed"] == False].iterrows():
    summary = row["summary"]
    if pd.isna(summary) or not isinstance(summary, str) or summary.strip() == "":
        df_pdf.at[idx, "doc_type_processed"] = True

# --- Find where to resume ---
start_idx = df_pdf[df_pdf["doc_type_processed"] == False].index.min()
if pd.isna(start_idx):
    print("All document types already processed.")
    start_idx = len(df_pdf)
else:
    print(f"Resuming document type detection from index {start_idx}.")

# --- Processing Loop ---
batch_counter = 0

for idx in range(start_idx, len(df_pdf)):
    row = df_pdf.loc[idx]
    summary = row["summary"]

    if pd.notna(summary) and isinstance(summary, str) and summary.strip():
        doc_type = call_llama_doc_type(summary)
        df_pdf.at[idx, "document_type"] = doc_type
        df_pdf.at[idx, "doc_type_processed"] = True
        batch_counter += 1

    if batch_counter >= batch_size:
        df_pdf.to_parquet(output_parquet_path, index=False)
        print(f"Saved after processing up to index {idx}.")
        batch_counter = 0

# --- Final Save ---
df_pdf.to_parquet(output_parquet_path, index=False)
print("Final save with document types complete.")


In [ ]:
import subprocess
import pandas as pd
import os

# --- Setup ---
ollama_path = r"C:\Users\terbe\AppData\Local\Programs\Ollama\ollama.exe"
model_name = "llama3.2"
input_parquet_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction_2.parquet"
output_parquet_path = r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\parquet_batches\pdf_text_extraction_3.parquet"

batch_size = 3

# --- Define function to call Llama 3.2 model ---
def call_llama_summarize(text):
    prompt = (
        f"Provide a summary from the provided text. "
        f"Give an overview providing details about the contents of the text, do not try to solve any problems. "
        f"ONLY output the summary. Do NOT include any introductions, explanations, or additional text.\n\n"
        f"{text}"
    )

    result = subprocess.run(
        [ollama_path, "run", model_name],
        input=prompt,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8"
    )

    if result.returncode != 0:
        print("Error:", result.stderr)
        return ""

    return result.stdout.strip()


# --- Load or create dataframe ---
if os.path.exists(output_parquet_path):
    df_pdf = pd.read_parquet(output_parquet_path)
    print("Loaded existing processed file.")
else:
    df_pdf = pd.read_csv(input_csv_path)
    df_pdf["llama_summary"] = ""
    df_pdf["llama_summary_processed"] = False
    print("Loaded fresh input CSV.")

# --- Ensure required columns ---
for col in ["llama_summary", "llama_summary_processed"]:
    if col not in df_pdf.columns:
        df_pdf[col] = "" if col == "llama_summary" else False

# --- Fix rows that have no text: mark them processed ---
for idx, row in df_pdf[df_pdf["llama_summary_processed"] == False].iterrows():
    text = row["summarized_text_cleaned"]
    if pd.isna(text) or not isinstance(text, str) or text.strip() == "":
        df_pdf.at[idx, "llama_summary_processed"] = True

# --- Find where to resume ---
start_idx = df_pdf[df_pdf["llama_summary_processed"] == False].index.min()
if pd.isna(start_idx):
    print("All rows already processed.")
    start_idx = len(df_pdf)
else:
    print(f"Resuming from index {start_idx}.")

# --- Processing Loop ---
batch_counter = 0

for idx in range(start_idx, len(df_pdf)):
    row = df_pdf.loc[idx]
    text_to_summarize = row["summarized_text_cleaned"]

    if pd.notna(text_to_summarize) and isinstance(text_to_summarize, str) and text_to_summarize.strip():
        summary = call_llama_summarize(text_to_summarize)
        df_pdf.at[idx, "llama_summary"] = summary
        df_pdf.at[idx, "llama_summary_processed"] = True
        batch_counter += 1

    # Save after every batch_size rows
    if batch_counter >= batch_size:
        df_pdf.to_parquet(output_parquet_path, index=False)
        print(f"Saved after processing up to index {idx}.")
        batch_counter = 0

# --- Final save ---
df_pdf.to_parquet(output_parquet_path, index=False)
print("Final save complete.")


In [ ]:
# !pip install textract

In [ ]:
df_text.to_csv(
    r"C:\Users\terbe\Desktop\Folder and Files Archiving -Box Sdk\extracted_text_preview.csv.gz",
    index=False,
    compression='gzip'
)


In [ ]:
# Preview 3 samples of each file type with extracted text
def print_samples(df, ext, n=3):
    print(f"\n--- {ext.upper()} FILE SAMPLES ---")
    subset = df[df['file_type'] == ext].dropna(subset=['raw_text']).head(n)
    for i, row in subset.iterrows():
        print(f"\nFile Path: {row['file_path']}")
        print(f"Text Preview:\n{row['raw_text'][:1000]}\n{'-'*60}")

# Print 3 samples each
for extension in ['.pdf', '.jpg', '.mhtml', '.eml', '.mbox', '.docx']:
    print_samples(df_text, extension)